# 03 — Local spikes versus SGD

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/03_bio_vs_artificial.ipynb)

Companion to §4 of *From Brains to Models*. Same two-dimensional points, two learners. A leaky integrate-and-fire perceptron updates each synapse from its own pre- and post-synaptic spike counts. A small MLP assigns error through a hidden layer by SGD.

On a linear pair of clouds both succeed. On a four-blob XOR only SGD does. That is the learning-rule row of the paper’s comparison table, at toy scale.

**Claim.** The elementary learning step is not the same object in a local spike-count rule and in SGD, even on identical points.

**Method.** LIF perceptron with a local delta rule vs a two-layer MLP; linear clouds and four-blob XOR. Seed 7.

**Result.** Both separate the linear pair; only SGD solves XOR.

**Limit.** Not a cortical column. Failure of this local rule is not proof that cortex cannot solve XOR.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **Leaky integrate-and-fire (LIF) neuron.** A minimal spiking model: inputs charge a membrane that leaks over time; when it crosses a threshold the neuron fires a spike and resets. The unit here learns from its own spike counts.
- **Local learning rule.** A synapse changes using only signals present at that synapse — pre- and post-synaptic activity — with no global error routed back from the output. Biologically plausible, computationally limited.
- **[Hebbian](https://en.wikipedia.org/wiki/Hebbian_theory) / delta rule.** "Cells that fire together wire together", here as a spike-count rule that nudges weights toward the correct classification.
- **Linear separability and XOR.** Two clouds a straight line can split are *linearly separable*; the four-blob XOR is not. Solving XOR requires combining features, which a single layer cannot do.
- **Multilayer perceptron (MLP) and SGD.** A network with a hidden layer trained by stochastic gradient descent, which assigns error through the hidden units by [backpropagation](https://en.wikipedia.org/wiki/Backpropagation) — the non-local step biology is not known to implement.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **LIF dynamics.** $\tau\dot{V} = -(V-V_{\text{rest}}) + I(t)$, with a spike when $V\ge V_\theta$ followed by a reset; learning uses pre/post spike counts over a window.
- **Local delta rule.** $\Delta w_i \propto (y^{\*}-\hat{y})\,x_i$ from local quantities only; equivalent to a single-layer perceptron, which converges iff the data are linearly separable.
- **XOR.** Not linearly separable; needs a hidden layer $h=\phi(W_1 x)$ then $\hat{y}=W_2 h$.
- **SGD / [backpropagation](https://en.wikipedia.org/wiki/Backpropagation).** Minimise cross-entropy by $\theta\leftarrow\theta-\eta\,\nabla_\theta\mathcal{L}$, with gradients passed through the hidden layer — a non-local credit-assignment signal.
- **Setup.** Two-dimensional points (linear clouds and four-blob XOR); 40 epochs; seed 7.

</details>

The same code writes `paper/figures/bio_vs_artificial.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook trains two learners — a spiking perceptron with a local rule and a small MLP with SGD — on the same two-dimensional points. The two blocks below define the recurring terms once, first in plain language and then in notation. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Spike and raster.** A spike is an all-or-none event; a raster plots spikes as dots over time. The spike-train panel shows the biological unit's output for one input point.
- **Credit assignment.** The problem of deciding which synapse to change after a mistake. A local rule guesses from nearby activity; backpropagation computes it exactly, but needs a global error signal.
- **Perceptron limit.** A single layer draws one boundary: it succeeds on the linear pair and fails on XOR — the classic demonstration that some problems require depth.
- **Hidden layer.** An intermediate set of units that build new features (such as "either, but not both"), letting the MLP bend the boundary XOR needs.
- **Epoch.** One full pass through the training points; learning curves are plotted over epochs.
- **What this is and is not.** The *learning-rule* row of the paper's brain-versus-model table at toy scale — not a cortical column, and not a claim that cortex cannot solve XOR by other means.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Two learners.** (i) A LIF perceptron with a local spike-count delta rule; (ii) a two-layer MLP trained by SGD / [backpropagation](https://en.wikipedia.org/wiki/Backpropagation).
- **Tasks.** Linear clouds (linearly separable) and four-blob XOR (not separable).
- **Objective.** Cross-entropy loss, with accuracy tracked per epoch.
- **Result.** Both separate the linear pair; only the MLP solves XOR.
- **Setup.** Seed 7; 40 epochs.
- **Limit.** Locality is the independent variable; the MLP's success shows what a non-local gradient buys, not that biology uses it.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.bio_vs_artificial import (
    SmallMLP,
    SpikingPerceptron,
    make_linear,
    make_xor,
    render_paper_figure,
    run_experiment,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## Two tasks, same plane

**Linear.** Two Gaussian clouds on the $x_1$ axis. A line suffices.

**XOR.** Four clouds at the corners of a square; opposite corners share a label. No straight line works. A hidden layer can draw the quadrants — *if* the label’s error is allowed to change the first-layer weights.

In [2]:
x_lin, y_lin = make_linear(n=200, seed=SEED)
x_xor, y_xor = make_xor(n=200, seed=SEED + 1)
print("linear", x_lin.shape, "class balance", y_lin.mean())
print("xor   ", x_xor.shape, "class balance", y_xor.mean())

linear (200, 2) class balance 0.5
xor    (200, 2) class balance 0.5


## A spike train for a point

Each coordinate becomes a Poisson process. Two leaky integrate-and-fire output cells race; the class is who spikes more. The weight $W_{ij}$ sees only the spike count of input $i$ and output $j$ — no error is sent to a hidden layer, because there is none.

In [3]:
bio = SpikingPerceptron(seed=SEED)
rng = np.random.default_rng(SEED)
pt = np.array([1.1, 0.0])
spikes = bio.encode(pt, rng)
print(f"point {pt}  input spike counts {spikes.sum(axis=0)}")

fig, ax = plt.subplots(figsize=(6.0, 1.8))
for i in range(2):
    times = np.where(spikes[:, i] > 0)[0]
    ax.scatter(times, np.full_like(times, i, dtype=float), s=18, c="0.2", linewidths=0)
ax.set_yticks([0, 1], ["$x_1$", "$x_2$"])
ax.set_xlabel("time bin")
ax.set_title("Poisson encoding of (1.1, 0)", loc="left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()

point [1.1 0. ]  input spike counts [13.  6.]


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49846/1845174679.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Train both learners

Forty epochs. The LIF net is online (one point, one local update). The MLP uses minibatches and backprop through a 16-unit ReLU layer.

In [4]:
results = run_experiment(seed=SEED, n_epochs=40)
for name, task in results.items():
    print(f"{name:7s}  LIF {task.acc_bio[-1]:.2f}   MLP {task.acc_mlp[-1]:.2f}")

linear   LIF 1.00   MLP 1.00
xor      LIF 0.50   MLP 1.00


## Paper figure

A: linear clouds and both decision regions. B: XOR. C–D: train accuracy. The same call writes the PDF the paper includes.

In [5]:
fig = render_paper_figure(results)
plt.show()
write_figure(results, root / "paper" / "figures")

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49846/1067417365.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/bio_vs_artificial.pdf
wrote <checkout>/paper/figures/bio_vs_artificial.png


PosixPath('<checkout>/paper/figures/bio_vs_artificial.pdf')

## What to notice

**The linear task is a control, not a victory.** Both learners draw a vertical cut. The LIF boundary is noisier because the encoding is Poisson. If the local rule had failed here, the XOR failure would be uninteresting.

**XOR is the credit-assignment schematic.** Opposite corners share a label. A synapse that only sees $x_1$ and its own postsynaptic cell cannot know that the *combination* with $x_2$ is what matters. SGD sends the output error back through the hidden units and carves the quadrants.

**This is not “brains cannot do XOR.”** Cortex is recurrent, layered, neuromodulated, and not a two-weight perceptron. The figure is the table’s learning-rule row: *local product of pre- and post-synaptic activity* versus *global gradient*. Approximations to backprop exist; they are extra machinery, not the perceptron above.

**Same objective-level kinship as notebooks 01–02, different mechanism.** Everyone here is fitting a classifier. The paper’s later claim is that mechanism still matters for energy, continual learning, and a body — even when representations later look alike.

## Why this example is here

This notebook is not a contest, and it is not a model of a cortical column. It is a controlled demonstration of the one fact §4 of the paper needs: **the elementary learning step is not the same object in a brain and in a foundation model**, even when both can be described as “neural nets.” Notebook 01 showed a global score on a string. Notebook 02 showed a local-ish Hebbian step on a patch. Here the two rules are given the *same points* so the difference is not the data.

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never trained a network or met a leaky integrate-and-fire cell. The two readings below are of the *same* figure.

### What each learner is asked to do

A point $x\in\mathbb{R}^2$ has a label $y\in\{0,1\}$.

The **LIF perceptron** turns each coordinate into a Poisson spike train (probability an increasing function of $x_i$). Two output cells integrate $W^\top s_t + b$ with a leak; they fire and reset when they cross a threshold. After $T$ bins one has input rates $r_\mathrm{in}$ and output rates $r_\mathrm{out}$. The update is local,

$$
\Delta W \propto r_\mathrm{in}\,(e)^\top,\qquad e = \mathbf{1}_y - r_\mathrm{out}.
$$

Synapse $(i,j)$ sees only the activity of input $i$ and output $j$. There is no hidden layer and no chain rule.

The **MLP** is $2\to 16\to 2$ with a ReLU hidden layer and a softmax. Training is minibatch SGD on cross-entropy. The hidden weights move by

$$
\delta h = (p - \mathbf{1}_y)\,W_2^\top,\qquad
\Delta W_1 \propto x^\top(\delta h \odot \mathbf{1}_{u>0}),
$$

which is exactly the step a local rule is not allowed: an error invented at the output is assigned to a unit that never produced that output.

**For the clinician and the neuroscientist.** A synapse in cortex “knows” the spikes that arrive and the spikes it helps to cause, plus slow neuromodulatory signals. It does not, as far as anyone has shown, know the gradient of a global loss with respect to every other synapse. That is why backprop-through-a-cortex is a research programme (feedback alignment, predictive coding as inference, target prop), not a default. The linear panel is a reflex or a one-layer association — both machines can do it. The XOR panel is a schematic of *context*: the meaning of $x_1$ depends on $x_2$, as a symptom’s meaning depends on the rest of the chart. A local first-order rule on the sensors misses that. A hidden layer with assigned credit does not. Cortex almost certainly has the second kind of capacity. The open question in §4 is whether it has it *by* backprop, or by something that only agrees with backprop at the level of representations (the Schrimpf / Goldstein citations in the paper).

**For the mathematically inclined.** The LIF net is a stochastic linear classifier in disguise. If $\phi$ is the (noisy) map from $x$ to spike counts, the decision is $\arg\max_k (W^\top \phi(x)+b)_k$, and $W$ is fit by a stochastic delta rule. The hypothesis class is essentially half-planes in $\phi$-space; for our $\phi$ (monotone per coordinate) that is still linearly separable problems in $x$. XOR is not in that class, so chance is the right asymptote, not a training failure. The MLP with one ReLU hidden layer is a universal approximator on compact sets in $\mathbb{R}^2$ already at modest width; SGD finds a quadrant partition because that partition is in the class *and* the gradient reaches $W_1$. No claim is made that this is the unique minimiser, or that LIF+$\Delta$ is the brain’s objective.

### How to read the panels

**A — linear, both succeed.** Left: the clouds. Middle: LIF region (Poisson encoding makes the cut a little ragged). Right: MLP region (a clean vertical). Panel C is the same fact as a curve.

**B — XOR, only SGD.** The local rule paints one colour: it has picked a side and cannot unpick it as a function of the other coordinate. The MLP draws the two diagonals’ worth of quadrants. Panel D: LIF at $1/2$, MLP at $1$.

**For the clinician and the neuroscientist.** If you only remember one picture, remember B. It is the difference between a test that looks at one lab value and a note that combines two. Local credit is the single value. Assigned credit is the combination.

**For the mathematically inclined.** B middle is a (noisy) linear discriminant in $x$. B right is a piecewise-linear partition; the kinks sit where ReLUs turn on. D is empirical risk on the training set, not a generalisation bound.

### What this is not

It is not STDP with millisecond pairing, not a recurrent cortical microcircuit, not Brian/Norse, and not a claim that Hebbian learning is weak in general (Oja, BCM, and competitive nets find useful features). It is not energy accounting: both toys run on a laptop. The table’s 20 W versus cluster-scale training is a fact about *people and foundation models*, not about these 200 points.

### What this has to do with being human

Section 2’s transformer and Section 3’s predictive-coding net can both be called neural. This notebook is why that word is doing too much work. A cortex that learns locally, sparsely, continually, on a power budget, in a body, is not a transformer that learns by a global gradient on a corpus and then freezes. Representations may still converge (the paper says so). The *cost* of getting them, and what can be updated later without a replay of the internet, need not. Later sections (dynamics, workspace, values) sit on that gap.

Change `SEED` or the blob variance and the ragged LIF cut will move. The claim should not: **same points; local spike-count credit versus a chain rule; XOR as the witness.**

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. Same points, two updates. Which of Marr’s rows agrees, and which does not?

2. XOR is the witness. What does failure of this local spike-count rule *not* prove about cortex?

3. SGD assigns credit through a hidden layer by [back-propagation](https://en.wikipedia.org/wiki/Backpropagation). A local [Hebbian](https://en.wikipedia.org/wiki/Hebbian_theory) count cannot. Where in a ward protocol do you already rely on a non-local “credit” that a single synapse would not see?

4. Leak is not depth. What does a membrane time-constant change that stacking layers changes, and why should those not be confused?

5. Representations may still converge across systems. If two networks match on a representation test, what learning-cost facts can still differ?

6. The 20-watt budget versus cluster-scale training is a fact about people and foundation models, not about these 200 points. What would a fair energy comparison have to measure?

7. The LIF rule can keep updating; the MLP here is a batch on a fixed set. Which clinical skill is closer to the first, and what breaks if you only have the second?

8. Separating two classes is a shared computational theory realized by two algorithms. State that sentence with the linear clouds in it, then with XOR.

9. Locality, sparsity, a body, a power budget: which of these is a hardware description, and which is already more than hardware ([embodiment](https://plato.stanford.edu/entries/embodied-cognition/))?

10. If both learners succeed on the linear clouds, why is that the uninteresting panel for the paper?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Lillicrap, T. P., Santoro, A., Marris, L., Akerman, C. J., and Hinton, G. (2020).** [*Backpropagation and the brain*](https://www.cs.toronto.edu/~hinton/absps/backpropandbrain.pdf). *Nature Reviews Neuroscience* 21, 335–346. Author PDF.
   - *(i) From brains to models.* Strict [backprop](https://en.wikipedia.org/wiki/Backpropagation) needs error signals that cortical feedback is unlikely to deliver. The opened review argues that feedback may instead induce activity whose *differences* locally approximate those signals. This notebook’s claim is narrower and prior: the elementary step we implemented (local spike-count vs SGD through a hidden layer) is not the same object.
   - *(ii) What AI reveals about being human.* If cortex learns XOR, it is not doing it by the perceptron in panel B. Approximations of backprop are still not a body, a 20 W budget, or continual learning without epochs. Mechanism ≠ organism.

2. **Yamins, D. L. K., and DiCarlo, J. J. (2016).** [*Using goal-driven deep learning models to understand sensory cortex*](https://doi.org/10.1038/nn.4244). *Nature Neuroscience* 19, 356–365.
   - *(i) From brains to models.* Goal-driven hierarchical convnets predict single-unit and population responses in higher visual cortex. The comparison licensed here is representational: a task-optimised hierarchy can be a model of ventral-stream responses. It is not a model of the learning rule in panel A.
   - *(ii) What AI reveals about being human.* A network that matches IT on object recognition still trains by global gradients on i.i.d. batches. Theme (ii) is everything Table 1 of the paper lists as divergent: spikes, locality, energy, a body that can fail.

3. **Doerig, A., et al. (2023).** [*The neuroconnectionist research programme*](https://konklab.fas.harvard.edu/Papers/Doerig_2023_NRN.pdf). *Nature Reviews Neuroscience* 24, 431–450.
   - *(i) From brains to models.* ANNs as a language for falsifiable brain theories. This notebook is one such theory stated so it can fail: *if* credit is only local on the sensors, XOR stays at chance. The MLP’s success is the extra that backprop buys, not a proof that cortex uses it.
   - *(ii) What AI reveals about being human.* A progressive programme can compare learning rules without claiming the winner is human. The remainder is the list the paper’s Table 1 already made: neuromodulation, continual protocol, embodiment.

4. **Kriegeskorte, N., Mur, M., and Bandettini, P. (2008).** [*Representational similarity analysis — connecting the branches of systems neuroscience*](https://www.frontiersin.org/articles/10.3389/neuro.06.004.2008/full). *Frontiers in Systems Neuroscience* 2, 4.
   - *(i) From brains to models.* RSA compares representational dissimilarity matrices so a model and a brain region can be related without a unit-to-voxel map. That is the honest grain for “the MLP matches cortex”: a second-order similarity, not a synapse.
   - *(ii) What AI reveals about being human.* A matching RDM is not a matching life. Dissimilarity structure can converge while the update rule, the energy, and the body diverge. Do not promote a matrix correlation into a person.

5. **Hebb, D. O. (1949).** [*The Organization of Behavior*](https://en.wikipedia.org/wiki/Hebbian_theory). Wiley. Entry via the opened Hebbian-theory page.
   - *(i) From brains to models.* The local slogan this notebook implements as spike counts: units that fire together wire together. [Hebbian](https://en.wikipedia.org/wiki/Hebbian_theory) plasticity is a pre/post coincidence, not an error assigned through a hidden layer.
   - *(ii) What AI reveals about being human.* Local coincidence is biologically native and computationally limited (panel B). Humans still solve XOR. The remainder is whatever cortex adds — recurrence, neuromodulation, a developmental trajectory — that this LIF perceptron does not have.

6. **Vaswani, A., et al. (2017).** [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). *NeurIPS* 30. arXiv:1706.03762.
   - *(i) From brains to models.* The other learner in the paper’s Table 1: dense, trained once by global gradients, frozen at inference. This notebook is the learning-rule row of that table, not the [transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) itself.
   - *(ii) What AI reveals about being human.* A successful MLP on four blobs is not a foundation model and not a cortex. Keep the scale of the claim in the same cell as the accuracy curve.


### Exercises

1. **XOR needs a hidden layer.** The notebook already trains both learners on the four-blob XOR. Report train accuracy for the spiking perceptron and for the MLP, then for an MLP with `n_hidden=1`. Which failure is about *local spike-count updates*, and which is about *having only a linear readout*?

<!-- solution-box: 03_1_xor_needs_a_hidden_layer.py -->
<details>
<summary><strong>Show solution</strong> — XOR needs a hidden layer</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/03_1_xor_needs_a_hidden_layer.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.bio_vs_artificial import SmallMLP, SpikingPerceptron, make_xor
x, y = make_xor(n=200, seed=SEED)
spk = SpikingPerceptron(seed=SEED)
mlp = SmallMLP(n_hidden=16, seed=SEED)
thin = SmallMLP(n_hidden=1, seed=SEED)
spk.train(x, y, n_epochs=30, seed=SEED)
mlp.train(x, y, n_epochs=30)
thin.train(x, y, n_epochs=30)
print(f"LIF perceptron (2→2, local delta)     acc {np.mean(spk.predict(x, seed=SEED) == y):.2f}")
print(f"MLP 2→16→2, SGD                       acc {np.mean(mlp.predict(x) == y):.2f}")
print(f"MLP 2→1→2, SGD                        acc {np.mean(thin.predict(x) == y):.2f}")
print(
    "\nThe thin MLP is still SGD, and it still cannot draw the XOR "
    "partition: one hidden unit is a single half-plane. The LIF net is "
    "a different failure: two output cells and a local count rule, no "
    "hidden layer to assign credit through. Same points, two reasons to "
    "miss. Neither miss is a synapse in cortex."
)
```

</details>
<!-- /solution-box -->


2. **Same points, different update.** On the linearly separable Gaussians both learners succeed. After training, scatter the two weight columns of the LIF net and the first-layer weights of the MLP (plot the 2-D input weights). The decision is similar; the objects that were updated are not. What was incremented in each case?

<!-- solution-box: 03_2_same_points_different_update.py -->
<details>
<summary><strong>Show solution</strong> — Same points, different update</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/03_2_same_points_different_update.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.bio_vs_artificial import SmallMLP, SpikingPerceptron, make_linear
x, y = make_linear(n=200, seed=SEED)
spk = SpikingPerceptron(seed=SEED)
mlp = SmallMLP(n_hidden=16, seed=SEED)
spk.train(x, y, n_epochs=25, seed=SEED)
mlp.train(x, y, n_epochs=25)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3))
for ax, pts, c, lab in (
    (axes[0], x[y == 0], "#4a7c8c", "class 0"),
    (axes[0], x[y == 1], "#c47a3a", "class 1"),
):
    ax.scatter(pts[:, 0], pts[:, 1], s=12, c=c, label=lab, alpha=0.7)
for k, col in enumerate(("#2c5f6e", "#9a4a2a")):
    w = spk.w[:, k]
    axes[0].arrow(0, 0, w[0], w[1], color=col, width=0.02, head_width=0.12)
    axes[0].set_title("LIF: two columns of W (2×2)", fontsize=9)
axes[0].legend(fontsize=8, frameon=False)
axes[0].set_aspect("equal")

im = axes[1].imshow(mlp.w1.T, cmap="coolwarm", aspect="auto")
axes[1].set_title("MLP: W1 (2 → 16), updated by SGD", fontsize=9)
axes[1].set_xlabel("input dim")
axes[1].set_ylabel("hidden unit")
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()
print(
    "LIF: each synapse saw a pre-count and a post-count. MLP: every hidden "
    "weight saw a scalar that had already been multiplied by the downstream "
    "error. Same point cloud; the elementary increment is not the same "
    "object. That is the §4 claim, not a verdict on which rule is 'more biological'."
)
```

</details>
<!-- /solution-box -->


3. **Leak is not depth.** The LIF cell has a `leak` (how fast $v$ forgets) and a `threshold`. Retrain on the linear task at `leak=0.05` and `leak=0.80`, same seed. Does accuracy move more than it did when you removed the MLP's hidden layer in exercise 1? What kind of difference is leak?

<!-- solution-box: 03_3_leak_is_not_depth.py -->
<details>
<summary><strong>Show solution</strong> — Leak is not depth</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/03_3_leak_is_not_depth.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.bio_vs_artificial import SpikingPerceptron, make_linear
x, y = make_linear(n=200, seed=SEED)
for leak in (0.05, 0.20, 0.80):
    net = SpikingPerceptron(leak=leak, seed=SEED)
    net.train(x, y, n_epochs=25, seed=SEED)
    acc = float(np.mean(net.predict(x, seed=SEED) == y))
    print(f"leak {leak:.2f}  train acc {acc:.2f}")
print(
    "\nLeak changes how long a spike is remembered inside one trial. It is "
    "a time-constant, not a layer. On a linearly separable cloud it barely "
    "moves the score; XOR would still be out of reach. Do not read a "
    "membrane parameter as architectural depth, and do not read this cell "
    "as cortex."
)
```

</details>
<!-- /solution-box -->
